# News sentiment as a trading signal: RavenPack, then FinBERT

Universe: the point-in-time top 1,000 U.S. stocks by market cap (list chosen at each year-end), 2020–2024.
Everything runs from local caches; no WRDS connection is needed.

**Notation**
- $d$: a trading day. $r_{i,d}$: CRSP total return of stock $i$ on day $d$ (close of $d-1$ to close of $d$).
- $s_k \in [-1, 1]$: the score of story $k$. RavenPack: the event sentiment score (ESS). FinBERT: $p_{pos} - p_{neg}$ from the headline.
- $S_{i,d}$: the mean score of the stories about stock $i$ assigned to day $d$; $n_{i,d}$: their number. $S_{i,d} = n_{i,d} = 0$ with no news.
- Signal: $\text{sent}_1(i,d) = S_{i,d}$ (one-day window, $L = 1$).
- IC at lag $k$: the Spearman rank correlation, across stocks with news on $d$, of $\text{sent}_1(i,d)$ and $r_{i,d+k}$, averaged over days. Only $k \ge 1$ is tradable.
- Quintiles: each day, stocks with news are sorted on $\text{sent}_1$ into $q_1$ (lowest) to $q_5$ (highest), equal-weighted. Long–short $= q_5 - q_1$.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sentlib import ravenpack as rp
from sentlib.wrds_io import DATA_DIR, load_membership
from sentlib.timing import assign_trading_day
from sentlib.signal import make_signal, load_signal
from backtest import wide_returns, forward_returns, daily_ic, ic_summary, quantile_portfolios, performance_table
from backtest.evaluate import wide_signal, rowwise_corr, lag_matrix

pd.set_option('display.width', 200, 'display.max_colwidth', 90)
TAG, START, END = 'top1000_2020-01-01_2024-12-31', '2020-01-01', '2024-12-31'

membership = load_membership(TAG)
# Cached from crsp.dsi (trading_calendar) and rp.build_link, starting 2019-12-19
calendar = pd.DatetimeIndex(pd.read_parquet(DATA_DIR / f'{TAG}_calendar.parquet')['date'])
link = pd.read_parquet(DATA_DIR / f'{TAG}_rp_link.parquet')
returns = pd.read_parquet(DATA_DIR / 'crsp_returns_2020-01-02_2025-02-14_1c5ead5a.parquet')
R, _, alive = wide_returns(returns)          # date x permno returns, and "can be bought on d"
fwd = forward_returns(returns, [1, 5])
signals = {name: load_signal(DATA_DIR / f'{TAG}_{name}_signal.parquet')
           for name in ['rp', 'rp_noprice', 'finbert_event', 'finbert_event_noprice', 'finbert_all_noprice']}
print(f"{membership.permno.nunique()} stocks, {len(link)} RavenPack links, {len(R)} return days")

## 0. The timing rule

A story counts toward the first trading day $d$ whose 15:30 New York time is after it.
The signal on $d$ is traded at the close of $d$, so the first return it can earn is $r_{i,d+1}$.

In [ ]:
ny = pd.Series(pd.to_datetime(['2023-05-24 15:29', '2023-05-24 16:20',
                               '2023-05-26 18:00', '2023-05-27 10:00'])).dt.tz_localize('America/New_York')
pd.DataFrame({'story time (New York)': ny, 'weekday': ny.dt.day_name(),
              'trading day d': assign_trading_day(ny.dt.tz_convert('UTC'), calendar)})

# Part 1: RavenPack

## 1.1 Stories and filters
One row per (story, company, detected event). Filters: relevance $\ge 90$ (the company is the subject),
a detected event (ESS not null), and novelty (`event_similarity_days` $\ge 1$: no similar story in the previous day).

In [ ]:
stories = rp.fetch_stories(None, link['rp_entity_id'].unique().tolist(), '2019-12-18', END,
                           cache_dir=DATA_DIR / 'rp_stories' / TAG)
steps = [('relevance >= 90', rp.StoryFilter(require_event=False, min_similarity_days=None)),
         ('+ event detected', rp.StoryFilter(min_similarity_days=None)),
         ('+ novel', rp.StoryFilter()),
         ('+ no price recaps', rp.StoryFilter(exclude_groups=('stock-prices',)))]
funnel = {'cached (relevance >= 75)': len(stories)}
funnel |= {label: len(rp.apply_filter(stories, f)) for label, f in steps}
pd.Series(funnel, name='rows').to_frame().style.format('{:,}')

## 1.2 One stock-day: NVIDIA's earnings, 24 May 2023
Results came out at 16:20 on Wednesday 24 May, after the cutoff, so they count toward Thursday 25 May.

In [ ]:
NVDA = 86580
ent = link.loc[link.permno == NVDA, 'rp_entity_id'].iloc[0]
s = rp.apply_filter(stories[stories.rp_entity_id == ent], rp.StoryFilter())
s['ny_time'] = s.timestamp_utc.dt.tz_localize('UTC').dt.tz_convert('America/New_York')
s['date'] = assign_trading_day(s.timestamp_utc, calendar)
ex = s[s.date.between('2023-05-24', '2023-05-25')]
ex[['ny_time', 'date', 'event_sentiment_score', 'event_similarity_days', 'group', 'headline']].iloc[::4]

In [ ]:
# Aggregation: S = mean ESS, n = count, by trading day; then the saved signal and the returns
agg = ex.groupby('date')['event_sentiment_score'].agg(S='mean', n='count')
sig = signals['rp'].query('permno == @NVDA').set_index('date')[['S', 'n']].loc['2023-05-23':'2023-05-26']
ret = R[NVDA].loc['2023-05-23':'2023-05-26'].rename('return r_d')
pd.concat([agg.add_suffix(' (recomputed)'), sig, ret], axis=1)

The +24% jump is $r_{d}$ on 25 May: it happened overnight, before anyone could trade on the 25 May signal.
The signal can only earn $r_{d+1}$ (26 May).

## 1.3 Coverage

In [ ]:
cov = pd.DataFrame({name: (signals[name].n_1 > 0).groupby(signals[name].date.dt.year).mean()
                    for name in ['rp', 'rp_noprice']})
cov.style.format('{:.1%}').set_caption('share of stock-days with news')

## 1.4 A trap: automated price recaps
Dow Jones and MarketWatch publish "X Stock Rises Wednesday" after the close. RavenPack tags them `stock-prices`.
They describe today's return, so a signal that includes them partly restates the past.

In [ ]:
f = rp.apply_filter(stories, rp.StoryFilter())
recap = f['group'].eq('stock-prices')
print(recap.groupby(f.timestamp_utc.dt.year).mean().round(3).rename('share of filtered stories'))
f.loc[recap & f.headline.str.contains(' Stock (?:Rises|Falls) ', na=False),
      ['timestamp_utc', 'event_sentiment_score', 'headline']].sample(6, random_state=1)

In [ ]:
def ic_by_lag(sig, lags=(-1, 0, 1, 2)):
    # Mean daily IC of sent_1 on d with r on d+k, and its t-stat
    X = wide_signal(sig, 'sent_1', index=R.index, columns=R.columns)
    out = {}
    for k in lags:
        ic = rowwise_corr(X, lag_matrix(R, alive, k)).dropna()
        out[k] = {'IC': ic.mean(), 't': ic.mean() / ic.std() * np.sqrt(len(ic))}
    return pd.DataFrame(out).T.rename_axis('lag k')

pd.concat({name: ic_by_lag(signals[name]) for name in ['rp', 'rp_noprice']}, axis=1).round(3)

Reading the table: $k = -1$ is yesterday's return, $k = 0$ today's (already in the price when we trade),
$k = 1$ is the first tradable return. Sentiment tracks returns up to the close, then nothing.

## 1.5 Backtest of `rp_noprice`

In [ ]:
def backtest(name, cost_bps=5):
    sig = signals[name]
    ic = ic_summary(daily_ic(sig, 'sent_1', fwd, 1), 1)
    res = quantile_portfolios(sig, 'sent_1', returns, cost_bps=cost_bps)
    daily = res['quantiles'].assign(long_short=res['long_short'],
                                    **{f'net {cost_bps} bps': res['long_short_net']})
    perf = performance_table(daily)

    fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
    perf.loc[res['quantiles'].columns, 'ann_ret'].plot.bar(ax=ax[0], color='#2a78d6')
    ax[0].set_title(f'{name}: annualized return by quintile')
    for c in ['long_short', f'net {cost_bps} bps']:
        ax[1].plot((1 + daily[c].fillna(0)).cumprod(), label=c)
    ax[1].set_title('long-short q5 - q1, growth of $1')
    ax[1].legend(frameon=False)
    plt.tight_layout()
    plt.show()
    print(f"IC(h=1) = {ic['mean']:.4f} (NW t = {ic['nw_t']:.2f}); "
          f"daily one-way turnover per leg = {res['turnover']['ls'].mean():.0%}")
    return perf[['ann_ret', 'ann_vol', 'sharpe', 'max_dd']].round(3)

backtest('rp_noprice')

# Part 2: FinBERT

FinBERT is BERT fine-tuned on financial text to classify a sentence as positive, negative or neutral.
We score the same RavenPack headlines, so the stories, timing and aggregation are unchanged. Only $s_k$ changes.

## 2.1 Scoring headlines live

In [ ]:
from finbert_score import load_model, score_texts, score_headlines

model, tokenizer, device = load_model()
demo = ['Nvidia 1Q Gaming Revenue Down 38% >NVDA',
        'Nvidia Poised for Record Sales as AI Demand Kicks In -- Update',
        'Nvidia Price Target Raised to $500.00/Share From $300.00 by TD Cowen',
        'Micron Technology Cut to Underperform From Buy by B of A Securities',
        'Xilinx Cut to Underperform From Neutral by B of A Securities',
        'Fox Corp. Cut to Underperform From Peer Perform by Wolfe Research',
        'NVIDIA Announces Financial Results for First Quarter Fiscal 2024']
out = score_texts(demo, model, tokenizer, device)
out.insert(0, 'headline', demo)
out['RavenPack ESS'] = [stories.loc[stories.headline == h, 'event_sentiment_score'].mean() for h in demo]
out.round(2)

FinBERT has no notion of a rating scale: the same "Cut to Underperform" template gets opposite scores
depending on the company and broker names. Across all cached headlines:

In [ ]:
cache = pd.read_parquet(DATA_DIR / 'finbert_scores' / 'ProsusAI__finbert.parquet')
pd.DataFrame({p: {'headlines': len(x := cache[cache.headline.str.contains(p, regex=False)]),
                  'share scored positive': (x.finbert_score > 0).mean()}
              for p in ['Cut to Underperform', 'Cut to Sell', 'Upgraded to Buy']}).T.round(2)

## 2.2 FinBERT vs RavenPack, story by story

In [ ]:
fb = rp.apply_filter(stories, rp.StoryFilter(exclude_groups=('stock-prices',)))
fb = fb.drop_duplicates(['rp_story_id', 'rp_entity_id']).reset_index(drop=True)
fb['finbert_score'] = score_headlines(fb['headline'])['finbert_score']     # all cached
print(f"{len(fb):,} stories; Pearson {fb.finbert_score.corr(fb.event_sentiment_score):.2f}, "
      f"Spearman {fb.finbert_score.corr(fb.event_sentiment_score, method='spearman'):.2f}")

fig, ax = plt.subplots(figsize=(5, 4.5))
ax.hexbin(fb.event_sentiment_score, fb.finbert_score, gridsize=40, bins='log', cmap='Blues')
ax.set_xlabel('RavenPack ESS')
ax.set_ylabel('FinBERT p_pos - p_neg')
plt.show()

In [ ]:
# Strong disagreements
cols = ['event_sentiment_score', 'finbert_score', 'headline']
pd.concat([fb[(fb.event_sentiment_score < -0.7) & (fb.finbert_score > 0.7)].sample(4, random_state=0)[cols],
           fb[(fb.event_sentiment_score > 0.7) & (fb.finbert_score < -0.7)].sample(4, random_state=0)[cols]]).round(2)

## 2.3 Same pipeline, new score
`make_signal` = assign trading day → daily mean and count → windows → universe membership. We rebuild
`finbert_event_noprice` and check it against the saved file.

In [ ]:
fb_sig = make_signal(fb, 'finbert_score', link, membership, calendar, [1], START, END, dedupe_story=True)
saved = signals['finbert_event_noprice']
print('rows match:', len(fb_sig) == len(saved),
      '| max |difference| in sent_1:', float((fb_sig.sent_1 - saved.sent_1).abs().max()))
print(f"correlation with rp_noprice (pooled, stock-days with news): "
      f"{fb_sig.sent_1[fb_sig.n_1 > 0].corr(signals['rp_noprice'].sent_1[fb_sig.n_1 > 0]):.2f}")

In [ ]:
ic_by_lag(signals['finbert_event_noprice']).round(3)

In [ ]:
backtest('finbert_event_noprice')

# 3. Summary across all five signals
`rp`, `finbert_event`: all novel event stories. `_noprice`: price recaps removed.
`finbert_all_noprice`: also stories without a RavenPack event (FinBERT can score any headline).

In [ ]:
lags = pd.concat({name: ic_by_lag(sig, (-1, 0, 1))['IC'] for name, sig in signals.items()}, axis=1).T
lags.columns = ['IC yesterday (k=-1)', 'IC today (k=0)', 'IC tomorrow (k=1)']
comp = pd.read_csv(DATA_DIR / 'backtest' / 'comparison_sent_1_news_q5_h1_equal.csv', index_col='signal')
summary = lags.join(comp[['t_1', 'ls_sharpe', 'ls_sharpe_5bps']].rename(
    columns={'t_1': 't (k=1)', 'ls_sharpe': 'LS Sharpe', 'ls_sharpe_5bps': 'LS Sharpe, 5 bps'}))
summary.round(3)

In [ ]:
ax = lags.plot.bar(figsize=(10, 3.8), color=['#9ab', '#2a78d6', '#eb6834'], rot=0)
ax.axhline(0, color='0.4', lw=0.8)
ax.set_title('Mean daily rank IC of sent_1 on d with the return on d+k')
plt.show()

**Takeaways**
1. Sentiment lines up with returns through the close of $d$ ($k \le 0$), then the IC is about zero:
   for large caps, news is priced before we can trade at the close.
2. Data hygiene first: price recaps make the signal partly a lagged return.
3. FinBERT and RavenPack agree only moderately (correlation about 0.4–0.5) and fail differently
   (FinBERT has no notion of a rating scale). Neither predicts next-day returns here.
4. About 85% of each leg turns over daily, so even 5 bps of costs makes the long-short strongly negative.

**Exercises**: longer windows and holding periods (`--windows 5 20`, `--holding 5`); smaller stocks;
returns measured from the next day's open.